<a href="https://colab.research.google.com/github/marveldw/big_data/blob/main/Pemrosesan_dan_Analisis_Data_Sales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pemrosesan data Dengan pyspark

Dalam latihan ini kita akan memproses data sales, yang memiliki kolom sebagai berikut :


| Nama Kolom | Tipe Data | Deskripsi | Kebutuhan Data |
| :--- | :--- | :--- | :--- |
| **TransactionID** | Kategorikal | ID unik untuk setiap transaksi penjualan. | mandatory |
| **CustomerID** | Kategorikal | ID untuk setiap pelanggan yang melakukan transaksi. | opsional |
| **ProductCategory** | Kategorikal | Jenis produk yang dijual. | mandatory |
| **SalesPrice** | Numerik | Nilai penjualan. | opsional |
| **Quantity** | Numerik | Jumlah unit produk yang dibeli. | mandatory |
| **TransactionDate** | Temporal (Tanggal) | Tanggal transaksi dilakukan (Format YYYY-MM-DD). | mandatory |
| **Region** | Kategorikal | Wilayah penjualan. | opsional |
| **DiscountCode** | Kategorikal | Kode diskon yang digunakan. | opsional |

In [1]:
#Download dataset
!wget https://raw.githubusercontent.com/urfie/datasets/refs/heads/main/tugas_pyspark/sales_toclean.csv

--2026-06-14 07:43:45--  https://raw.githubusercontent.com/urfie/datasets/refs/heads/main/tugas_pyspark/sales_toclean.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3194227 (3.0M) [text/plain]
Saving to: ‘sales_toclean.csv’

sales_toclean.csv   100%[===================>]   3.05M  --.-KB/s    in 0.07s   

2026-06-14 07:43:46 (46.2 MB/s) - ‘sales_toclean.csv’ saved [3194227/3194227]



Inisialisasi spark session

In [2]:
from pyspark.sql import SparkSession

# Inisiasi SparkSession
spark = SparkSession.builder.appName("SalesDataProcessing").getOrCreate()

In [3]:
#loading data ke dataframe
df = spark.read.csv("sales_toclean.csv", header=True, inferSchema=True)

Melakukan *Quick Checking* untuk melihat sekilas profil dataset : jumlah baris, skema, dan summary statistik.

In [4]:
#tampilkan skema
df.printSchema()

root
 |-- TransactionID: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- ProductCategory: string (nullable = true)
 |-- SalesPrice: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- TransactionDate: date (nullable = true)
 |-- Region: string (nullable = true)
 |-- DiscountCode: string (nullable = true)



In [5]:
#Tampilkan 10 baris pertama
df.show(10)

+-------------+----------+---------------+----------+--------+---------------+-------+------------+
|TransactionID|CustomerID|ProductCategory|SalesPrice|Quantity|TransactionDate| Region|DiscountCode|
+-------------+----------+---------------+----------+--------+---------------+-------+------------+
|       TE0001|      C630|           FOOD|     14.63|       9|     2024-12-27|CENTRAL|      DISC02|
|       TE0002|      C176|           HOME|     130.0|       9|     2024-12-29|CENTRAL|      Disc01|
|       TE0003|      C805|           HOME|     92.47|      10|     2024-12-25|   EAST|      Disc01|
|       TE0004|      C918|    Electronics|     119.3|       8|     2024-12-22|   West|      DISC02|
|       TE0005|     C1062|    Electronics|    228.37|       1|     2024-12-26|   EAST|      DISC04|
|       TE0006|      C492|           Toys|      26.0|       9|     2024-12-29|   EAST|      DISC02|
|       TE0007|      C981|    Electronics|    261.27|      10|     2024-12-22|   EAST|      DISC04|


In [6]:
# tampilkan jumlah baris
# hints: gunakan fingsi count()

print("Jumlah baris: ", df.count())

Jumlah baris:  63000


In [7]:
# tampilkan jumlah kolom
# hinst: gunakan properti column

print("Jumlah kolom: ", len(df.columns))

Jumlah kolom:  8


In [8]:
# tampilkan rangkuman statistik
df.describe().show()

+-------+-------------+----------+---------------+-----------------+-----------------+-------+------------+
|summary|TransactionID|CustomerID|ProductCategory|       SalesPrice|         Quantity| Region|DiscountCode|
+-------+-------------+----------+---------------+-----------------+-----------------+-------+------------+
|  count|        63000|     63000|          63000|            63000|            63000|  60942|       48507|
|   mean|         NULL|      NULL|           NULL|73.45803904761843|5.516126984126984|   NULL|        NULL|
| stddev|         NULL|      NULL|           NULL|73.40392478137521|2.875832206385795|   NULL|        NULL|
|    min|       T00001|      C100|          BOOKS|              9.0|                1|CENTRAL|      DISC01|
|    max|       TN3000|      C999|    electronics|            450.0|               10|   West|      Disc01|
+-------+-------------+----------+---------------+-----------------+-----------------+-------+------------+



**Jawablah pertanyaan berikut ini**

In [9]:
# berapa nilai NULL pada kolom ProductCategory

df.filter(df.ProductCategory.isNull()).count()

0

In [10]:
# berapa nilai NULL pada kolom SalesPrice

df.filter(df.SalesPrice.isNull()).count()

0

In [11]:
# berapa nilai NULL pada kolom Quantity

df.filter(df.Quantity.isNull()).count()

0

In [12]:
# berapa nilai NULL pada kolom Region

df.filter(df.Region.isNull()).count()

2058

In [13]:
# berapa nilai NULL pada kolom DiscountCode

df.filter(df.DiscountCode.isNull()).count()

14493

## Menangani nilai hilang/NULL

Untuk menangani nilai yang hilang, ada dua hal yang dapat dilakukan :
1. Hapus data yang memiliki nilai NULL di salah satu/lebih kolom mandatory
2. Berikan nilai default untuk kolom yang opsional


In [14]:
# Hapus data
df_no_null = df.dropna(subset=["ProductCategory", "SalesPrice", "Quantity"])

Beri nilai default untuk kolom-kolom opsional:
- Kolom SalesPrice : ganti NULL dengan 0
- Kolom Region dan CustomerID : ganti NULL dengan "UNKNOWN"
- Kolom DiscountCode : ganti NULL dengan "NO_DISCOUNT"

In [15]:
from pyspark.sql.functions import col, lit, when

df_imputed = df_no_null.withColumn(
    "DiscountCode",
    when(col("DiscountCode").isNull(),
         lit("NO_DISCOUNT")).otherwise(col("DiscountCode")))

In [16]:
# Isi kolom Region
df_imputed = df_imputed.withColumn(
    "Region",
    when(col("Region").isNull(),
         lit("UNKNOWN")).otherwise(col("Region")))

In [17]:
# Isi kolom CustomerID
df_imputed = df_imputed.withColumn(
    "CustomerID",
    when(col("CustomerID").isNull(),
         lit("UNKNOWN")).otherwise(col("CustomerID")))

In [18]:
# Isi kolom SalesAmount
df_imputed = df_imputed.fillna(0, subset=["SalesPrice"])

## Pertanyaan


**Ada berapa baris data setelah dibersihkan nilai NULL-nya?**

In [19]:
# lengkapi kode berikut

df_imputed.count()

63000

**Berapa rata-rata nilai SalesAmount dan Quantity?**

*Hint : gunakan fungsi describe()*

In [20]:
# lengkapi kode berikut

df_imputed.describe(["SalesPrice", "Quantity"]).show()

+-------+-----------------+-----------------+
|summary|       SalesPrice|         Quantity|
+-------+-----------------+-----------------+
|  count|            63000|            63000|
|   mean|73.45803904761843|5.516126984126984|
| stddev|73.40392478137521|2.875832206385795|
|    min|              9.0|                1|
|    max|            450.0|               10|
+-------+-----------------+-----------------+



## Standarisasi Nilai Kolom

Untuk kolom `ProductCategory`, `Region`, dan `DiscountCode`, tampilkan nilai unik tiap kolom untuk mengetahui apakah ada nilai yang tidak standard dan apa yang perlu dilakukan untuk melakukan standarisasi.

In [21]:
df_imputed.select("ProductCategory").distinct().show()

+---------------+
|ProductCategory|
+---------------+
|           HOME|
|          BOOKS|
|           TOYS|
|    Electronics|
|           FOOD|
|    ELECTRONICS|
|    electronics|
|           Toys|
|       CLOTHING|
+---------------+



In [22]:
from pyspark.sql.functions import upper, trim

df_standard = df_imputed.withColumn("ProductCategory",
                                    trim(upper(col("ProductCategory"))))

In [23]:
df_standard.select("ProductCategory").distinct().show()

+---------------+
|ProductCategory|
+---------------+
|           HOME|
|          BOOKS|
|           TOYS|
|           FOOD|
|    ELECTRONICS|
|       CLOTHING|
+---------------+



In [24]:
df_standard.select("Region").distinct().show()

+-------+
| Region|
+-------+
|CENTRAL|
|UNKNOWN|
|Central|
|   East|
|   West|
|   EAST|
+-------+



In [25]:
df_standard = df_standard.withColumn("Region",
                                    trim(upper(col("Region"))))

In [26]:
df_standard.select("Region").distinct().show()

+-------+
| Region|
+-------+
|   WEST|
|CENTRAL|
|UNKNOWN|
|   EAST|
+-------+



In [27]:
df_standard.select("DiscountCode").distinct().show()

+------------+
|DiscountCode|
+------------+
|      DISC01|
|      Disc01|
|      DISC05|
| NO_DISCOUNT|
|      DISC02|
|      DISC04|
+------------+



In [28]:
df_standard = df_standard.withColumn("DiscountCode",
                                    trim(upper(col("DiscountCode"))))

In [29]:
df_standard.select("DiscountCode").distinct().show()

+------------+
|DiscountCode|
+------------+
|      DISC01|
|      DISC05|
| NO_DISCOUNT|
|      DISC02|
|      DISC04|
+------------+



## Bersihkan data duplikat

Tidak boleh ada data yang memiliki kombinasi nilai CustomerID, ProductCategory, SalesAmount, Quantity, TransactionDate, dan Region yang sama.

Hapus data duplikat tersebut

In [30]:
df_nodup = df_standard.dropDuplicates(subset=["CustomerID", "ProductCategory",
                                              "SalesPrice", "Quantity",
                                              "TransactionDate", "Region"])

In [31]:
df_nodup.count()

63000

## Pertanyaan

Jawab pertanyaan berikut ini :


**1. Ada berapa jenis diskon?**

In [32]:
# lengkapi kode berikut

df_nodup.select("DiscountCode").distinct().count()

5

**2. Ada berapa region?**

In [33]:
# lengkapi kode berikut

df_nodup.select("Region").distinct().count()

4

**3. Ada berapa jenis produk?**

In [34]:
# lengkapi kode berikut

df_nodup.select("ProductCategory").distinct().count()

6

**4. Berapa jumlah record setelah dibersihkan dari duplikasi?**

In [35]:
# lengkapi kode berikut

df_nodup.count()

63000

## Enrichment

Dalam tahap ini kita akan melakukan 2 hal :
1. Membuat kolom baru dari kolom yang sudah ada, yaitu membuat kolom baru dengan rumus : SalesPrice * Quantity
2. Menggabungkan dengan data referensi
3. Mendapatkan nilai diskon dengan rumus : SalesAmount * Rate

Data referensi dapat diunduh di : https://raw.githubusercontent.com/urfie/datasets/refs/heads/main/tugas_pyspark/discount.csv

In [36]:
df_amount = df_nodup.withColumn("SalesAmount",
                                 col("SalesPrice") * col("Quantity"))

In [37]:
!wget https://raw.githubusercontent.com/urfie/datasets/refs/heads/main/tugas_pyspark/discount.csv

--2026-06-14 07:44:40--  https://raw.githubusercontent.com/urfie/datasets/refs/heads/main/tugas_pyspark/discount.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 132 [text/plain]
Saving to: ‘discount.csv’

discount.csv        100%[===================>]     132  --.-KB/s    in 0s      

2026-06-14 07:44:40 (2.40 MB/s) - ‘discount.csv’ saved [132/132]



In [38]:
df_discount = spark.read.csv("discount.csv", header=True, inferSchema=True)

In [39]:
df_discount.show()

+------------+------------+-----+
|DiscountCode| Description| Rate|
+------------+------------+-----+
|      DISC01|Family Promo| 0.03|
|      DISC02|     Promo 1| 0.02|
|      DISC03|     Promo 2| 0.02|
|      DISC05|     Promo 3| 0.05|
|      DISC04|    New Year|0.025|
+------------+------------+-----+



In [40]:
df_enriched = df_amount.join(df_discount,
                              on="DiscountCode",
                              how="left")

In [41]:
df_enriched = df_enriched.withColumn("DiscountAmount",
                                      col("SalesAmount") * col("Rate"))

In [42]:
df_enriched.createOrReplaceTempView("sales_data")

## Pertanyaan Analisis Data



**1. Kategori Produk mana yang nilai penjualannya paling tinggi?**

In [43]:
# lengkapi kode berikut dengan menggunakan perintah SQL

spark.sql("""
SELECT ProductCategory, SUM(SalesAmount) AS TotalSalesAmount
FROM sales_data
GROUP BY ProductCategory
ORDER BY TotalSalesAmount DESC
""").show()

+---------------+--------------------+
|ProductCategory|    TotalSalesAmount|
+---------------+--------------------+
|    ELECTRONICS|1.1974283589999974E7|
|           HOME|   5462267.729999995|
|       CLOTHING|   4092170.449999995|
|           TOYS|  1905009.5300000017|
|          BOOKS|  1349334.6399999997|
|           FOOD|   866546.8300000005|
+---------------+--------------------+



**2. Berapa total nilai penjualan setelah diskon untuk kategori BOOKS? (bulatkan sampai 2 angka di belakang koma)**

In [44]:
# lengkapi kode berikut dengan menggunakan perintah SQL

spark.sql("""
SELECT ROUND(SUM(SalesAmount - COALESCE(DiscountAmount, 0)), 2) AS TotalSalesAfterDiscount
FROM sales_data
WHERE ProductCategory = 'BOOKS'
""").show()

+-----------------------+
|TotalSalesAfterDiscount|
+-----------------------+
|             1316980.98|
+-----------------------+



**3. Identifikasi 3 CustomerID yang menghasilkan TotalRevenue paling tinggi (selain UNKNOWN).**

In [45]:
# lengkapi kode berikut dengan menggunakan perintah SQL

spark.sql("""
SELECT CustomerID, SUM(SalesAmount - COALESCE(DiscountAmount, 0)) AS TotalRevenue
FROM sales_data
WHERE CustomerID != 'UNKNOWN'
GROUP BY CustomerID
ORDER BY TotalRevenue DESC
LIMIT 3
""").show()

+----------+------------------+
|CustomerID|      TotalRevenue|
+----------+------------------+
|      C573|48448.933950000006|
|     C1087|        43819.5445|
|     C1068|       43528.87405|
+----------+------------------+



**4. Program diskon mana yang paling banyak digunakan? (sebutkan namanya)**

In [46]:
# lengkapi kode berikut dengan menggunakan perintah SQL

spark.sql("""
SELECT DiscountCode, COUNT(*) AS UsageCount
FROM sales_data
WHERE DiscountCode != 'NO_DISCOUNT'
GROUP BY DiscountCode
ORDER BY UsageCount DESC
LIMIT 1
""").show()

+------------+----------+
|DiscountCode|UsageCount|
+------------+----------+
|      DISC01|     16853|
+------------+----------+



**5. Region mana yang paling banyak menggunakan diskon berdasar jumlah transaksi?**

In [47]:
# lengkapi kode berikut dengan menggunakan perintah SQL

spark.sql("""
SELECT Region, COUNT(*) AS TransactionCount
FROM sales_data
WHERE DiscountCode != 'NO_DISCOUNT'
GROUP BY Region
ORDER BY TransactionCount DESC
LIMIT 1
""").show()

+------+----------------+
|Region|TransactionCount|
+------+----------------+
|  WEST|           15668|
+------+----------------+



**6. Region mana yang paling banyak menggunakan diskon berdasar besarnya nilai yang terdiskon (amount)?**

In [48]:
# lengkapi kode berikut dengan menggunakan perintah SQL

spark.sql("""
SELECT Region, SUM(DiscountAmount) AS TotalDiscountedAmount
FROM sales_data
WHERE DiscountCode != 'NO_DISCOUNT'
GROUP BY Region
ORDER BY TotalDiscountedAmount DESC
LIMIT 1
""").show()

+------+---------------------+
|Region|TotalDiscountedAmount|
+------+---------------------+
|  WEST|   208766.92869999993|
+------+---------------------+



**7. Di wilayah East, kategori produk CLOTHING, kode diskon apa yang paling sering digunakan, dan berapa rata-rata nilai penjualan sebelum diskon untuk kode tersebut?**

In [49]:
# lengkapi kode berikut dengan menggunakan perintah SQL

spark.sql("""
SELECT DiscountCode, COUNT(*) AS Frequency, AVG(SalesAmount) AS AvgSalesBeforeDiscount
FROM sales_data
WHERE Region = 'EAST' AND ProductCategory = 'CLOTHING'
GROUP BY DiscountCode
ORDER BY Frequency DESC
LIMIT 1
""").show()

+------------+---------+----------------------+
|DiscountCode|Frequency|AvgSalesBeforeDiscount|
+------------+---------+----------------------+
|      DISC01|     1120|           330.6669375|
+------------+---------+----------------------+

